##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multimodal Semantic Search with Gemini Embedding 2

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Multimodal_Semantic_Search_Embedding2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

## Overview

[`gemini-embedding-2-preview`](https://ai.google.dev/gemini-api/docs/models/gemini-embedding-2-preview) is the first multimodal embedding model in the Gemini API. It maps **text, images, video, audio, and PDFs** into a single unified vector space, enabling cross-modal semantic search — for example, retrieving images with a text query, or finding text documents that match an image.

This notebook shows how to build a multimodal semantic search index using the Gemini API directly (no third-party vector database required) and demonstrates three retrieval patterns:

1. **Text → Text**: Classic semantic document retrieval with `task_type` optimization.
2. **Text → Image**: Use a natural-language query to find the most relevant image from a collection.
3. **Image → Text**: Use an image as the query to retrieve matching text descriptions.

All three patterns rely on the same unified embedding space, so no special adapters or cross-modal bridges are needed.

## Setup

In [ ]:
%pip install -q -U google-genai numpy

### Configure your API key

To run this notebook, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. See [Authentication](../quickstarts/Authentication.ipynb) for details.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

EMBEDDING_MODEL = "gemini-embedding-2-preview"  # @param ["gemini-embedding-2-preview"] {allow-input: true}

### Helper: cosine similarity search

A lightweight in-memory search function. For production use cases, swap this out for a vector database (see the [Chroma](chromadb/Vectordb_with_chroma.ipynb), [Qdrant](qdrant/Qdrant_similarity_search.ipynb), or [Weaviate](weaviate/personalized_description_with_weaviate_and_gemini_api.ipynb) examples in this cookbook).

In [ ]:
import numpy as np


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def top_k(query_embedding: list[float], index: list[dict], k: int = 3) -> list[dict]:
    """Return the top-k most similar entries from the index."""
    scored = [
        {**entry, "score": cosine_similarity(query_embedding, entry["embedding"])}
        for entry in index
    ]
    return sorted(scored, key=lambda x: x["score"], reverse=True)[:k]

## Part 1: Text → Text semantic search

Embed a collection of text documents with `RETRIEVAL_DOCUMENT` task type, then query with `RETRIEVAL_QUERY`. Using the correct `task_type` improves retrieval quality because the model produces vectors optimized for asymmetric retrieval (short query vs. longer document).

In [ ]:
documents = [
    {
        "id": "doc_0",
        "title": "Photosynthesis",
        "text": (
            "Photosynthesis is the process by which green plants, algae, and some bacteria "
            "convert light energy into chemical energy stored as glucose. It takes place "
            "primarily in the chloroplasts using carbon dioxide and water, releasing oxygen "
            "as a byproduct."
        ),
    },
    {
        "id": "doc_1",
        "title": "The Water Cycle",
        "text": (
            "The water cycle describes the continuous movement of water on, above, and below "
            "the surface of the Earth. Key stages include evaporation, condensation, "
            "precipitation, and collection. Solar energy drives evaporation while gravity "
            "drives precipitation."
        ),
    },
    {
        "id": "doc_2",
        "title": "Newton's Laws of Motion",
        "text": (
            "Newton's three laws describe the relationship between a body and the forces "
            "acting upon it. The first law states that an object at rest stays at rest unless "
            "acted on by a net force. The second law relates force, mass, and acceleration "
            "(F = ma). The third law states that every action has an equal and opposite reaction."
        ),
    },
    {
        "id": "doc_3",
        "title": "DNA and Heredity",
        "text": (
            "DNA (deoxyribonucleic acid) carries the genetic instructions for the development, "
            "functioning, and reproduction of all known living organisms. Genes are segments "
            "of DNA that encode proteins, and heredity is the passing of traits from parents "
            "to offspring through these genes."
        ),
    },
    {
        "id": "doc_4",
        "title": "Black Holes",
        "text": (
            "A black hole is a region of spacetime where gravity is so strong that nothing — "
            "not even light — can escape once it crosses the event horizon. Black holes form "
            "when massive stars collapse at the end of their life cycle and are detected "
            "indirectly through their effects on surrounding matter."
        ),
    },
]

print(f"Embedding {len(documents)} documents...")

text_index = []
for doc in documents:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=doc["text"],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    )
    text_index.append({**doc, "embedding": response.embeddings[0].values})

print(f"Index built. Embedding dimension: {len(text_index[0]['embedding'])}")

In [ ]:
query = "How do plants produce energy from sunlight?"  # @param {type: "string"}

query_embedding = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=query,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
).embeddings[0].values

results = top_k(query_embedding, text_index)

print(f"Query: {query!r}\n")
for r in results:
    print(f"[{r['score']:.4f}] {r['title']}")
    print(f"  {r['text'][:120]}...\n")

## Part 2: Text → Image search

Build an index of images, then retrieve the best match for a natural-language query.

Because `gemini-embedding-2-preview` places all modalities in the same vector space, the text query embedding is directly comparable to image embeddings — no cross-modal bridge needed.

In [ ]:
# Download a small set of labeled images
image_urls = {
    "tiger.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Bengal_tiger_%28Panthera_tigris_tigris%29_female_3_crop.jpg/640px-Bengal_tiger_%28Panthera_tigris_tigris%29_female_3_crop.jpg",
    "dog.jpg": "https://upload.wikimedia.org/wikipedia/commons/3/3b/BlkStdSchnauzer2.jpg",
    "cat.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Tabby_cat_with_visible_nictitating_membrane.jpg/500px-Tabby_cat_with_visible_nictitating_membrane.jpg",
    "beach.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/640px-Good_Food_Display_-_NCI_Visuals_Online.jpg",
    "forest.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/640px-24701-nature-natural-beauty.jpg",
}

for filename, url in image_urls.items():
    print(f"Downloading {filename}...")
    !wget -q -O {filename} "{url}"

In [ ]:
import mimetypes

print("Building image index...")

image_index = []
for filename in image_urls:
    mime_type = mimetypes.guess_type(filename)[0] or "image/jpeg"
    with open(filename, "rb") as f:
        image_bytes = f.read()

    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type=mime_type)
        ],
    )
    image_index.append(
        {"filename": filename, "embedding": response.embeddings[0].values}
    )
    print(f"  Embedded {filename}")

print("\nImage index ready.")

In [ ]:
text_query = "a large wild cat with stripes"  # @param {type: "string"}

query_embedding = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=text_query,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
).embeddings[0].values

results = top_k(query_embedding, image_index)

print(f"Query: {text_query!r}\n")
for r in results:
    print(f"[{r['score']:.4f}] {r['filename']}")

In [ ]:
from IPython.display import Image, display

top_result = results[0]
print(f"Top result: {top_result['filename']} (score: {top_result['score']:.4f})")
display(Image(filename=top_result["filename"], width=300))

## Part 3: Image → Text search

Use an image as the query to find the most semantically similar text description from a collection. This demonstrates the full cross-modal nature of the unified embedding space — the same `top_k` function works regardless of which modality is the query and which is the index.

In [ ]:
# A collection of short text descriptions to search over
descriptions = [
    {"id": "d0", "text": "A fierce predatory big cat native to Asia, known for its orange fur and black stripes."},
    {"id": "d1", "text": "A loyal domestic pet, bred over thousands of years as a companion to humans."},
    {"id": "d2", "text": "A small independent feline often kept as a house pet, known for its agility."},
    {"id": "d3", "text": "A tropical sandy shore with waves lapping at the coast under bright sunlight."},
    {"id": "d4", "text": "A dense woodland filled with tall trees, undergrowth, and dappled light."},
]

print("Embedding text descriptions...")
desc_index = []
for desc in descriptions:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=desc["text"],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    )
    desc_index.append({**desc, "embedding": response.embeddings[0].values})

print("Done.")

In [ ]:
# Use the dog image as the query
image_query_file = "dog.jpg"  # @param {type: "string"}

with open(image_query_file, "rb") as f:
    image_bytes = f.read()

mime_type = mimetypes.guess_type(image_query_file)[0] or "image/jpeg"

image_query_embedding = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=[types.Part.from_bytes(data=image_bytes, mime_type=mime_type)],
).embeddings[0].values

results = top_k(image_query_embedding, desc_index)

print(f"Image query: {image_query_file!r}\n")
display(Image(filename=image_query_file, width=200))
print()
for r in results:
    print(f"[{r['score']:.4f}] {r['text']}")

## Bonus: Truncated embeddings (Matryoshka)

`gemini-embedding-2-preview` supports [Matryoshka Representation Learning (MRL)](https://arxiv.org/abs/2205.13147), which means you can reduce the embedding dimension with minimal quality loss. This is useful when storage or latency is a constraint.

The default dimension is **3072**. Common smaller sizes: 1536, 768, 256.

In [ ]:
text = "What causes auroras in the night sky?"

for dim in [3072, 1536, 768, 256]:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=dim),
    )
    print(f"output_dimensionality={dim:>5}: vector length = {len(response.embeddings[0].values)}")

## Next steps

- Scale to larger collections with a vector database: [Chroma](chromadb/Vectordb_with_chroma.ipynb), [Qdrant](qdrant/Qdrant_similarity_search.ipynb), [Weaviate](weaviate/personalized_description_with_weaviate_and_gemini_api.ipynb)
- Add a generative step to build a full RAG pipeline: [Talk to documents with embeddings](Talk_to_documents_with_embeddings.ipynb)
- Combine with Haystack for a production-ready pipeline: [Cross-modal Retrieval with Haystack](haystack/Gemini_Embedding_Haystack_Crossmodal_Retrieval.ipynb)
- Learn about all `task_type` options: [Embeddings quickstart](../quickstarts/Embeddings.ipynb)